# 📊 Chapter 5: Bayes' Theorem
*Tier 1 — Foundations | All Tracks*

> **🎬 Hook:** A test for a rare disease is 99% accurate. You test positive. Are you doomed?
> Bayes' Theorem says: probably not — and gives us the exact number.

**🎯 Objectives:** Understand prior, likelihood, posterior. Apply Bayes to real problems.

## 📖 Section 1 — Concept Review

Bayes' Theorem is the engine of rational belief updating:

$$P(H|E) = \frac{P(E|H) \cdot P(H)}{P(E)}$$

| Term | Name | Meaning |
|------|------|---------|
| P(H) | **Prior** | Your belief BEFORE seeing evidence |
| P(E\|H) | **Likelihood** | How probable is the evidence IF H is true? |
| P(H\|E) | **Posterior** | Your updated belief AFTER seeing evidence |
| P(E) | **Normalizing constant** | Total probability of seeing this evidence |

The denominator expands as:
$$P(E) = P(E|H)P(H) + P(E|\neg H)P(\neg H)$$

### Intuition: Bayes as Belief Updating
```
PRIOR → [Evidence arrives] → POSTERIOR
  ↑                               ↑
What we believed              What we should believe now
before seeing data             given the data
```

## 🧮 Section 2 — Math Walkthrough

### Cookie Jar Example
Two jars: Jar 1 has 30 chocolate + 10 vanilla. Jar 2 has 20 of each.
You pick a jar at random and draw a chocolate cookie. Which jar is it probably from?

- P(Jar 1) = P(Jar 2) = 0.5 (equal prior)
- P(Choc | Jar 1) = 30/40 = 0.75
- P(Choc | Jar 2) = 20/40 = 0.50

$$P(\text{Jar 1} | \text{Choc}) = \frac{0.75 \times 0.5}{0.75 \times 0.5 + 0.50 \times 0.5} = \frac{0.375}{0.625} = 0.6$$

After drawing a chocolate cookie, P(Jar 1) goes from 50% to **60%**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
np.random.seed(42)

# Bayes' Theorem calculator
def bayes(prior, likelihood_h, likelihood_not_h):
    numerator = likelihood_h * prior
    denominator = likelihood_h * prior + likelihood_not_h * (1 - prior)
    return numerator / denominator

# Medical test: vary prevalence
prevalences = np.linspace(0.001, 0.5, 300)
posterior = [bayes(p, 0.99, 0.01) for p in prevalences]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(prevalences, posterior, lw=3, color='#e74c3c')
axes[0].axhline(0.5, color='gray', linestyle='--', lw=1.5)
axes[0].set_xlabel('Prior P(Disease)')
axes[0].set_ylabel('Posterior P(Disease | Positive)')
axes[0].set_title('🏥 Bayes: How Base Rate Shapes Posterior', fontweight='bold')
axes[0].fill_between(prevalences, posterior, alpha=0.2, color='#e74c3c')

# Sequential Bayesian updating
prior = 0.5
evidence_seq = [0.75, 0.75, 0.50, 0.75, 0.75]  # likelihood each time chocolate drawn
posteriors = [prior]
for lik in evidence_seq:
    prior = bayes(prior, lik, 0.50)
    posteriors.append(prior)

axes[1].plot(posteriors, 'bo-', markersize=10, lw=2.5)
axes[1].axhline(0.5, color='gray', linestyle='--', lw=1.5, label='50/50 prior')
axes[1].set_xlabel('Number of Cookie Draws')
axes[1].set_ylabel('P(Jar 1 | Observations)')
axes[1].set_title('🍪 Sequential Bayesian Updating', fontweight='bold')
axes[1].set_ylim(0.4, 1.0)
axes[1].legend()

plt.tight_layout()
plt.savefig('ch05_bayes.png', dpi=150, bbox_inches='tight')
plt.show()

print("Medical test (99% accurate, disease prevalence 0.1%):")
print(f"  P(Disease | Positive) = {bayes(0.001, 0.99, 0.01):.3f}")
print()
print("Cookie jar after 5 chocolate draws:")
print(f"  P(Jar 1) = {posteriors[-1]:.3f}")

## 🔬 Section 3 — Simulation

In [ ]:
# Simulate Bayesian spam filter updating
np.random.seed(42)
n = 10000
is_spam = np.random.random(n) < 0.3

# 5 features: each has different P(feature|spam) and P(feature|legit)
features = {
    'FREE':    {'p_spam': 0.7, 'p_legit': 0.05},
    'WINNER':  {'p_spam': 0.6, 'p_legit': 0.02},
    'Meeting': {'p_spam': 0.05,'p_legit': 0.5},
    'Invoice': {'p_spam': 0.1, 'p_legit': 0.3},
    'Urgent':  {'p_spam': 0.4, 'p_legit': 0.1},
}

# Sequential update for one email with features: FREE=yes, WINNER=yes, Meeting=no
prior = 0.3
updates = [('Prior', prior)]

for fname, vals in list(features.items())[:3]:
    present = True if fname in ['FREE', 'WINNER'] else False
    if present:
        prior = bayes(prior, vals['p_spam'], vals['p_legit'])
    updates.append((f"After '{fname}'", prior))

print("📧 Bayesian Spam Filter - Sequential Updates")
for name, p in updates:
    bar = '█' * int(p * 40)
    print(f"  {name:<20} P(spam) = {p:.3f}  {bar}")

## 📊 Section 4 — Visualization: Prior → Posterior

In [ ]:
# Visualize prior vs posterior distributions
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

scenarios = [
    {'prior': 0.01, 'lik_h': 0.95, 'lik_nh': 0.05, 'title': 'Rare disease
(Prior=1%)'},
    {'prior': 0.20, 'lik_h': 0.95, 'lik_nh': 0.05, 'title': 'Moderate prevalence
(Prior=20%)'},
    {'prior': 0.50, 'lik_h': 0.95, 'lik_nh': 0.05, 'title': 'Common condition
(Prior=50%)'},
]

for ax, s in zip(axes, scenarios):
    posterior_val = bayes(s['prior'], s['lik_h'], s['lik_nh'])
    categories = ['No Disease', 'Disease']
    prior_vals = [1 - s['prior'], s['prior']]
    post_vals  = [1 - posterior_val, posterior_val]
    x = np.arange(2)
    ax.bar(x - 0.2, prior_vals,  0.35, label='Prior', color='#3498db', alpha=0.7)
    ax.bar(x + 0.2, post_vals,   0.35, label='Posterior', color='#e74c3c', alpha=0.7)
    ax.set_xticks(x); ax.set_xticklabels(categories)
    ax.set_title(s['title'] + f'
→ P(D|+) = {posterior_val:.2f}', fontweight='bold', fontsize=10)
    ax.set_ylim(0, 1.15)
    ax.legend(fontsize=8)

plt.suptitle("Bayes' Theorem: Prior vs Posterior (given positive test)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('ch05_prior_posterior.png', dpi=150, bbox_inches='tight')
plt.show()

## 📂 Section 5 — Real Exercise: A/B Testing with Bayes

You're testing a new website button. Your prior belief: P(new is better) = 0.5.
After 100 visits: new gets 65 clicks, old gets 50 clicks. Update your belief.

In [ ]:
from scipy import stats

np.random.seed(42)
# Beta-Binomial Bayesian A/B test
# Prior: Beta(1,1) = uniform (no preference)
# Observed: new=65 clicks out of 100, old=50 out of 100

alpha_prior, beta_prior = 1, 1  # uniform prior

# Posterior for each variant
new_clicks, new_visits = 65, 100
old_clicks, old_visits = 50, 100

new_posterior = stats.beta(alpha_prior + new_clicks, beta_prior + new_visits - new_clicks)
old_posterior = stats.beta(alpha_prior + old_clicks, beta_prior + old_visits - old_clicks)

x = np.linspace(0.3, 0.9, 300)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(x, new_posterior.pdf(x), lw=3, color='#27ae60', label=f'New button (65/100)')
ax.plot(x, old_posterior.pdf(x), lw=3, color='#e74c3c', label=f'Old button (50/100)')
ax.fill_between(x, new_posterior.pdf(x), alpha=0.2, color='#27ae60')
ax.fill_between(x, old_posterior.pdf(x), alpha=0.2, color='#e74c3c')
ax.axvline(0.65, color='#27ae60', linestyle='--', alpha=0.5)
ax.axvline(0.50, color='#e74c3c', linestyle='--', alpha=0.5)
ax.set_xlabel('Click-Through Rate')
ax.set_ylabel('Posterior Density')
ax.set_title('🖱️ Bayesian A/B Test: Posterior Distributions', fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('ch05_ab_test.png', dpi=150, bbox_inches='tight')
plt.show()

# P(new > old) via Monte Carlo
samples_new = new_posterior.rvs(100_000)
samples_old = old_posterior.rvs(100_000)
p_new_better = (samples_new > samples_old).mean()
print(f"P(new button is better) = {p_new_better:.3f}")
print(f"Expected CTR (new): {new_posterior.mean():.3f}")
print(f"Expected CTR (old): {old_posterior.mean():.3f}")

## ✏️ Section 6 — Practice Problems

**P1:** A factory has 2 machines. Machine A produces 60% of output with 2% defect rate.
Machine B produces 40% with 5% defect rate. You pick a random defective item.
P(from Machine A)?

**P2:** You take a COVID test with sensitivity=85%, specificity=95%.
Population prevalence=3%. You test positive. What is P(you have COVID)?

---
<details><summary>💡 Solutions</summary>

**P1:** P(A) = 0.6, P(defect|A) = 0.02, P(defect|B) = 0.05
P(defect) = 0.6×0.02 + 0.4×0.05 = 0.012 + 0.020 = 0.032
P(A|defect) = (0.02×0.6)/0.032 = **0.375**

**P2:** P(D|+) = (0.85×0.03)/(0.85×0.03 + 0.05×0.97) = 0.0255/0.0740 = **34.5%**
</details>

In [ ]:
# Solutions
print("P1:", bayes(0.6, 0.02, 0.05))
print("P2:", bayes(0.03, 0.85, 0.05))

## 🎯 Recap
1. **Prior × Likelihood → Posterior** — the core of Bayesian reasoning.
2. Base rate (prior) is crucial — ignoring it causes the False Positive Paradox.
3. Bayesian updating is sequential — each new piece of evidence refines your belief.

**Next:** [Chapter 6 — Random Variables: PMF, PDF, and CDF]